In [ ]:
from fastmcp import FastMCP

mcp=FastMCP(
    name="First Server"
)

print("MCP Server created successfully!")

@mcp.tool
def add(a:int,b:int)-> int:
    """ this can accept two numbers and return the sum of those numbers """
    return a+b

@mcp.tool
def mul(a:int,b:int=1):
    """ this can accept two numbers and return the multiplication of those numbers """
    return a*b



MCP Server created successfully!


### Connect with FastMCP Client

In [39]:
from fastmcp import Client

async with Client(mcp) as client:
    print("protocall version",client.protocol_version)
    tools= await client.list_tools()
    print("print the tool")
    result= await client.call_tool("add",{"a":12,"b":3})
    print(result.data)

protocall version 2026-07-28
print the tool
15


### All about the Mcp Tools- Context / Tasks / Auth

In [ ]:

from fastmcp import FastMCP, Client
import json


async def show(server: FastMCP, tool_name: str):
    async with Client(server) as client:
        tools = await client.list_tools()
        tool = None
        for t in tools:
            if t.name == tool_name:
                tool = t
                break

        if tool is None:
            print(f"Tool '{tool_name}' not found.")
            return

        print(f"Name: {tool.name}")
        print(f"Description: {tool.description}")
        print(json.dumps(tool.inputSchema, indent=2))


In [49]:
await show(mcp,"mul")

Name: mul
Description: this can accept two numbers and return the multiplication of those numbers 
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "a": {
      "type": "integer"
    },
    "b": {
      "default": 1,
      "type": "integer"
    }
  },
  "required": [
    "a"
  ]
}


In [53]:
from typing import Literal

@mcp.tool
async def export(table:str,format:Literal["csv","excel","json"]="csv"):
    """Export a table data into specific format"""
    return "Export done"

[10/29/26 13:11:12] WARNING  Component already exists: tool:export@                           ]8;id=15196370;file:///home/satyam/.local/lib/python3.10/site-packages/fastmcp/server/providers/local_provider/local_provider.py\local_provider.py]8;;\:]8;id=15196371;file:///home/satyam/.local/lib/python3.10/site-packages/fastmcp/server/providers/local_provider/local_provider.py#192\192]8;;\

In [54]:
await show(mcp,"export")

Name: export
Description: Export a table data into specific format
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "table": {
      "type": "string"
    },
    "format": {
      "default": "csv",
      "enum": [
        "csv",
        "excel",
        "json"
      ],
      "type": "string"
    }
  },
  "required": [
    "table"
  ]
}


### A Pydantic model as an argument

In [58]:
from pydantic import BaseModel, Field


class Shipment(BaseModel):
    order_id: str = Field(description="This is order id placed by the user...")
    quantity: int
    express: bool = False


@mcp.tool
def book_shipment(shipment: Shipment) -> dict:
    """Book a shipment for an order"""
    return {"booked": shipment.order_id, "qty": shipment.quantity}


[10/29/26 14:48:39] WARNING  Component already exists: tool:book_shipment@                    ]8;id=15196376;file:///home/satyam/.local/lib/python3.10/site-packages/fastmcp/server/providers/local_provider/local_provider.py\local_provider.py]8;;\:]8;id=15196377;file:///home/satyam/.local/lib/python3.10/site-packages/fastmcp/server/providers/local_provider/local_provider.py#192\192]8;;\

In [59]:
await show(mcp,"book_shipment")

Name: book_shipment
Description: Book a shipment for an order
{
  "type": "object",
  "additionalProperties": false,
  "properties": {
    "shipment": {
      "properties": {
        "order_id": {
          "description": "This is order id placed by the user...",
          "type": "string"
        },
        "quantity": {
          "type": "integer"
        },
        "express": {
          "default": false,
          "type": "boolean"
        }
      },
      "required": [
        "order_id",
        "quantity"
      ],
      "type": "object"
    }
  },
  "required": [
    "shipment"
  ]
}


Exception / Error Handeling

In [ ]:
from fastmcp.exceptions import ToolError

@mcp.tool
def new_charges_amount(amount:float):
    "Charge on a Card"
    if amount < 100:
        raise ToolError("Amount value should not be less then 100 rs.")
    return "Charges done..."

[10/29/26 14:57:16] WARNING  Component already exists: tool:charges_amount@                   ]8;id=15196394;file:///home/satyam/.local/lib/python3.10/site-packages/fastmcp/server/providers/local_provider/local_provider.py\local_provider.py]8;;\:]8;id=15196395;file:///home/satyam/.local/lib/python3.10/site-packages/fastmcp/server/providers/local_provider/local_provider.py#192\192]8;;\

In [76]:
async with Client(mcp) as client:
    res=await client.call_tool("new_charges_amount",{"amount":20},raise_on_error=False)
    print(res)

CallToolResult(content=[TextContent(type='text', text="Unknown tool: 'new_charges_amount'", annotations=None, meta=None)], structured_content=None, meta={'io.modelcontextprotocol/serverInfo': {'name': 'First Server', 'version': '4.1.4'}}, data=None, is_error=True)


#### `Context` - Talking back to the client while server is running..

In [81]:
from fastmcp import FastMCP,Context
import asyncio
ctxserver=FastMCP("cts-demo")

@ctxserver.tool
async def send_mails(emails:list[str],context: Context)->str:
    """Send Emails to many student at a time"""
    totalStudents=len(emails)
    ### one email at a time, 2 sec to complete
    for i in range(totalStudents):
        email=emails[i]
        await context.report_progress(progress=i+1,total=totalStudents,message=f"Sending email to :{email}")
        await asyncio.sleep(2)
    return "Email Sent to all the student"
        

In [82]:
async def watch_email_process(progess,total,message):
    print(message)

async with Client(ctxserver) as client:
    res=await client.call_tool("send_mails",{"emails":["prateek","nikhil","ajay"]},progress_handler=watch_email_process)
    print(res)
    
    
    

Sending email to :prateek
Sending email to :nikhil
Sending email to :ajay
CallToolResult(content=[TextContent(type='text', text='Email Sent to all the student', annotations=None, meta=None)], structured_content={'result': 'Email Sent to all the student'}, meta={'fastmcp': {'wrap_result': True}, 'io.modelcontextprotocol/serverInfo': {'name': 'cts-demo', 'version': '4.1.4'}}, data='Email Sent to all the student', is_error=False)


##### Long Running Task

In [ ]:
import asyncio

from fastmcp import FastMCP, Client
from fastmcp.dependencies import Progress
from fastmcp_tasks import TasksExtension
from fastmcp.utilities.tasks import TaskConfig

mcp = FastMCP("Task Server")

mcp.add_extension(TasksExtension())
mcp = FastMCP("Task Server")

mcp.add_extension(TaskExtension())


@mcp.tool()
async def slow_computation(
    email_counts: int,
    progress=Progress()
) -> str:

    await progress.set_total(email_counts)

    for i in range(email_counts):
        await asyncio.sleep(1)

        await progress.set_message(
            f"Sending Email {i + 1}/{email_counts}"
        )

        await progress.increment()

    return f"Completed {email_counts} emails"

In [ ]:
async with Client(mcp) as client:

    task = await call_tool_task(
        client,
        "slow_computation",
        {"email_counts": 10}
    )

    print("Task ID:", task.task_id)